In [9]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

### Simulations Parameter ##############
n = 100
seed = 42
n_sim = 100

jk_ab_calc = False
boot_var_calc = True
ijk_std_calc = True
train_models = True


########## Varying Parameters ##########
tau = 12
B_RF = int(n*0.7  * 2)
B_1 = 5

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': seed, 'tau': tau}

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  seed,
                'bootstrap':     True,  }

X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

######## Start Simulation   ########
with ProcessPoolExecutor() as executor:
    
    ### Array to store the results
    portion_events_after_cut_train = np.zeros(n_sim)
    portion_censored_after_cut_train = np.zeros(n_sim)
    portion_no_events_after_cut_train = np.zeros(n_sim)
    portion_events_after_cut_test = np.zeros(n_sim)
    portion_censored_after_cut_test = np.zeros(n_sim)
    portion_no_events_after_cut_test = np.zeros(n_sim)
    wb_mse_ipcw = np.zeros(n_sim)
    wb_cindex_ipcw = np.zeros(n_sim)
    wb_y_pred_X_point = np.zeros(n_sim)
    rf_mse_ipcw = np.zeros(n_sim)
    rf_y_pred_X_point = np.zeros(n_sim)
    ijk_var_pred_X_point = np.zeros(n_sim)
    bootstrap_std_pred_X_point = np.zeros(n_sim)

    futures = [
        executor.submit(
            simulation,
            seed=seed+i,
            tau=tau, 
            data_generation_weibull_parameters=params_data_creation,
            params_rf=params_rf, 
            X_pred_point=X_erwartung,
            B_first_level = B_1,
            boot_std_calc = boot_var_calc,
            ijk_std_calc = ijk_std_calc,
            train_models = train_models,
            jk_ab_calc = jk_ab_calc
        )
        for i in range(n_sim)
    ]

    for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
        _portion_events_after_cut_train, _portion_censored_after_cut_train, _portion_no_events_after_cut_train, \
         _portion_events_after_cut_test, _portion_censored_after_cut_test, _portion_no_events_after_cut_test, \
        _wb_mse_ipcw, _wb_cindex_ipcw, _wb_y_pred_X_point, \
        _rf_mse_ipcw, _rf_y_pred_X_point, _ijk_var_pred_X_point, _bootstrap_std_pred_X_point  = future.result()

        #Event-Stats Results
        portion_events_after_cut_train[i] = _portion_events_after_cut_train
        portion_censored_after_cut_train[i] = _portion_censored_after_cut_train
        portion_no_events_after_cut_train[i] = _portion_no_events_after_cut_train
        portion_events_after_cut_test[i] = _portion_events_after_cut_test
        portion_censored_after_cut_test[i] = _portion_censored_after_cut_test
        portion_no_events_after_cut_test[i] = _portion_no_events_after_cut_test
        
        #Evaluation Results
        wb_mse_ipcw[i] = _wb_mse_ipcw
        wb_cindex_ipcw[i] = _wb_cindex_ipcw
        rf_mse_ipcw[i] = _rf_mse_ipcw

        #Prediction Results
        wb_y_pred_X_point[i] = _wb_y_pred_X_point[0]
        rf_y_pred_X_point[i] = _rf_y_pred_X_point[0]

        # Standard Deviation Estimates
        ijk_var_pred_X_point[i]  = _ijk_var_pred_X_point
        bootstrap_std_pred_X_point[i] = _bootstrap_std_pred_X_point

Simulations: 100%|██████████| 100/100 [00:15<00:00,  6.56simulation/s]


In [10]:
def print_results():
    print('Evaluation Results:')
    print(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw):.4f}')
    print(f'WB MSE IPCW: {np.mean(wb_mse_ipcw):.4f}')
    print(f'RF MSE IPCW: {np.mean(rf_mse_ipcw):.4f}')
    print('\n')

    print('Prediction Results:')
    print(f'WB Y_pred: {np.mean(wb_y_pred_X_point):.4f}')
    print(f'RF Y_pred: {np.mean(rf_y_pred_X_point):.4f}')
    print('\n')

    print('Standard Deviation:')
    print(f'WB EMP-STD:                 {np.std(wb_y_pred_X_point):.4f}\n')
    print(f'RF EMP-STD:                 {np.std(rf_y_pred_X_point):.4f}')


    non_neg_var_ijk_est = ijk_var_pred_X_point.copy()
    non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
    # erst die wurzel ziehen und dann den mittelwert bilden
    np.mean(np.sqrt(  non_neg_var_ijk_est    ))
    print(f'IJK STD (for RF) Mean-est : {np.mean(np.sqrt(  non_neg_var_ijk_est    )):.4f}')

    non_neg_var_ijk_est = bootstrap_std_pred_X_point.copy()
    non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
    # erst die wurzel ziehen und dann den mittelwert bilden
    np.mean(np.sqrt(  non_neg_var_ijk_est    ))
    print(f'Boot STD (for RF) Mean-est :{np.mean(  bootstrap_std_pred_X_point    ):.4f}')

print_results()

Evaluation Results:
WB C-Index IPCW: 0.6068
WB MSE IPCW: 0.0721
RF MSE IPCW: 0.0851


Prediction Results:
WB Y_pred: 0.9551
RF Y_pred: 0.9601


Standard Deviation:
WB EMP-STD:                 0.0221

RF EMP-STD:                 0.1021
IJK STD (for RF) Mean-est : 0.0198
Boot STD (for RF) Mean-est :0.0541
